# Entrenamiento de Nuevo Modelo ML

Este notebook te permite entrenar nuevos modelos de Machine Learning para el bot de trading.

## Pasos:
1. Cargar y preparar datos
2. Crear features (variables)
3. Entrenar modelo
4. Evaluar rendimiento
5. Exportar modelo para producción

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 1. Cargar Datos

Carga tus datos históricos de criptomonedas

In [ ]:
# Ejemplo: cargar datos desde CSV
# df = pd.read_csv('../../datos_historicos.csv')

# O usar la API de Binance para obtener datos frescos
from binance.client import Client
from datetime import datetime, timedelta, timezone
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("BINANCE_API_KEY")
api_secret = os.getenv("BINANCE_SECRET")
client = Client(api_key, api_secret)

# Configuración
symbol = 'SOLUSDC'
period = 730  # días de historia
hour = 10  # hora de referencia

print(f"Descargando datos de {symbol}...")

## 2. Crear Features

Genera las variables técnicas para el modelo

In [ ]:
def combinations(l):
    l_final = []
    n = len(l)
    for i in range(n-1):
        for j in range(i+1, n):
            l_final.append((l[i], l[j]))
    return l_final

rachas = [1, 3, 7, 14, 30, 90, 180, 360]
dinamicas = combinations(rachas)

# Crear returns básicos
# df['high_return'] = df['high'] / df['open']
# df['low_return'] = df['low'] / df['open']
# df['close_return'] = df['close'] / df['open']
# df['volatility'] = df['high_return'] - df['low_return']

# Crear medias móviles
# for r in rachas:
#     df[f'high_mean_{r}'] = df['high_return'].rolling(r).mean()
#     df[f'low_mean_{r}'] = df['low_return'].rolling(r).mean()
#     df[f'close_mean_{r}'] = df['close_return'].rolling(r).mean()
#     df[f'volatility_mean_{r}'] = df['volatility'].rolling(r).mean()

# Crear dinámicas
# for d in dinamicas:
#     a, b = d
#     df[f'high_dinamica_mean_{a}_{b}'] = (df[f'high_mean_{a}'] / df[f'high_mean_{b}']) - 1
#     # ... más features

print("Features creadas")

## 3. Crear Variable Objetivo

Define qué quieres predecir (ej: si el precio subirá más de X%)

In [ ]:
# Ejemplo: predecir si el low del día siguiente será menor que el open
# df['target'] = (df['low'].shift(-1) / df['open'].shift(-1)) < 1.0

# O predecir rentabilidad mínima
# threshold = 1.015  # 1.5% mínimo
# df['target'] = (df['low'].shift(-1) / df['open'].shift(-1)) >= threshold

print("Variable objetivo creada")

## 4. Entrenar Modelo

In [ ]:
# Preparar datos
# df_clean = df.dropna()
# X = df_clean.drop(columns=['target', 'date', 'open', 'high', 'low', 'close'])
# y = df_clean['target']

# Split train/test
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# Entrenar modelo
# model = GradientBoostingClassifier(
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=5,
#     random_state=42
# )

# model.fit(X_train, y_train)

print("Modelo entrenado")

## 5. Evaluar Modelo

In [ ]:
# Predicciones
# y_pred = model.predict(X_test)
# y_pred_proba = model.predict_proba(X_test)[:, 1]

# Métricas
# print(classification_report(y_test, y_pred))
# print(f"ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Curva ROC
# fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
# plt.figure(figsize=(10, 6))
# plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_pred_proba):.2f})')
# plt.plot([0, 1], [0, 1], 'k--')
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curve')
# plt.legend()
# plt.show()

## 6. Optimizar Umbral de Decisión

In [ ]:
# Encontrar el mejor umbral
# best_threshold = 0.5
# best_score = 0

# for threshold in np.arange(0.3, 0.9, 0.05):
#     y_pred_custom = (y_pred_proba >= threshold).astype(int)
#     # Calcular tu métrica personalizada (ej: precisión, F1, etc.)
#     # score = ...
#     # if score > best_score:
#     #     best_score = score
#     #     best_threshold = threshold

# print(f"Mejor umbral: {best_threshold}")
# print(f"Mejor score: {best_score}")

## 7. Exportar Modelo para Producción

**IMPORTANTE**: Guarda el modelo en la carpeta `../pipeline/models/`

In [ ]:
# Nombre del modelo
model_name = 'mi_nuevo_modelo'

# Guardar modelo
# with open(f'../pipeline/models/{model_name}.pkl', 'wb') as f:
#     pickle.dump(model, f)

# Guardar columnas (features)
# with open(f'../pipeline/models/{model_name}_columns.pkl', 'wb') as f:
#     pickle.dump(X.columns.tolist(), f)

# Guardar configuración de umbral
# umbral_config = [
#     1.015,  # umbral de precio objetivo (ej: 1.5% ganancia)
#     0.985,  # umbral de stop loss (ej: -1.5% pérdida)
#     best_threshold  # umbral de probabilidad del modelo
# ]

# with open(f'../pipeline/models/{model_name}_umbral.pkl', 'wb') as f:
#     pickle.dump(umbral_config, f)

print(f"✅ Modelo exportado a ../pipeline/models/{model_name}.*")
print("")
print("📝 Siguiente paso: Actualizar config.py con:")
print(f"""MODEL_CONFIG = {{
    'model_path': 'models/{model_name}.pkl',
    'columns_path': 'models/{model_name}_columns.pkl',
    'threshold_path': 'models/{model_name}_umbral.pkl',
    ...
}}""")

## 8. Importancia de Variables (Opcional)

In [ ]:
# feature_importance = pd.DataFrame({
#     'feature': X.columns,
#     'importance': model.feature_importances_
# }).sort_values('importance', ascending=False)

# plt.figure(figsize=(12, 8))
# plt.barh(feature_importance['feature'].head(20), feature_importance['importance'].head(20))
# plt.xlabel('Importancia')
# plt.title('Top 20 Variables Más Importantes')
# plt.gca().invert_yaxis()
# plt.tight_layout()
# plt.show()